# SSF2 RL — Exploration & Data Collection

This notebook connects to the instrumented SSF2 build, lets you poke at the
observation/action space, drive the character with scripted inputs, and record
trajectories you can later use for supervised learning (behavioral cloning) or
to sanity-check your own RL implementations.

**Prerequisites** (run once, from the repo root):
```bash
cd /Users/cachemiss/Documents/projects/reflash2-fork/reflash2
.venv/bin/pip install -e python          # makes ssf2_rl importable
```

**The game auto-launches.** `env.reset()` starts the game itself if it isn't
already running — no manual terminal step needed. Requires `AIR_SDK_HOME` set,
or an AIR SDK at `~/Developer/AIRSDK*`. Game output is logged to
`.macos/adl.log`; `ssf2_rl.stop_game()` quits a game started this way. To
start it manually instead:
```bash
AIR_SDK_HOME="$HOME/Developer/AIRSDK_51.3.3" bash tools/macos/run_macos.sh
```

**One env, one hierarchy.** Every slot is declared with a single class
(`Agent` / `Human` / `CPU` / bots), and everything runs through `env`:
`env.reset(players=..., stage=...)` → `env.run(frames)` for real-time
evaluation → `env.step()` to step through one frame at a time.

Select the repo's `.venv` as the notebook kernel (Cmd+Shift+P → "Notebook: Select Kernel" → `.venv`).

In [2]:
# Player declarations are controller declarations:
# Agent / ZeroBot / FollowBot / ScriptedBot are Python-driven;
# Human and CPU remain native game controllers.
from ssf2_rl.bots import Agent, ZeroBot, FollowBot, ScriptedBot
from ssf2_rl.players import CPU, Human, Character, Stage
from ssf2_rl import NOOP, LEFT, RIGHT, DOWN, SPECIAL, ATTACK
from ssf2_rl.env import SSF2Env

try:
    env.close()
except NameError:
    pass
env = SSF2Env(step_timeout=5.0)  # minimal JSON is the compatibility default

## 1. Connect & reset (programmatic match setup)

`reset()` restarts the match in-game and takes over the bot slots. Every slot
is declared with one class — the declaration IS the controller:

```python
env.reset(players={1: Agent("marth"), 2: CPU("samus", level=0)}, stage="battlefield")
```

- `Agent` — driven by `env.step(action)` (the RL path)
- `Human` — you play it in the game window
- `CPU` — the in-game AI at a level
- `ZeroBot` / `FollowBot` / `ScriptedBot` / `PolicyBot` — Python bots

`env.describe_matchup()` prints exactly who controls each slot. After reset,
use `env.run(frames)` to watch it play in real time, or loop `env.step()` to
step through one frame at a time.

In [11]:
# Programmatic match setup: declare every slot and the stage in reset().
obs, info = env.reset(
    players={1: Agent(Character.Marth), 2: ZeroBot(Character.ZeroSuitSamus)},
    stage=Stage.bf,
)
print(env.describe_matchup())
print("\nframe:", info["frame"], "| me:", info["me"]["name"], "| opp:", info["opp"]["name"])

# ZeroBot's latest zero mask is held until Python replaces it; it cannot fall
# through to the backing native level-9 AI while Python is idle.
full = env.request_full_state()
zss = next(char for char in full["chars"] if char["id"] == 2)
assert zss["controls"] == 0
print("ZeroBot held controls:", zss["controls"])

[ssf2_rl] Launching SSF2 via /Users/cachemiss/Developer/AIRSDK_51.3.3/bin/adl (log: /Users/cachemiss/Documents/projects/reflash2-fork/reflash2/.macos/adl.log) ...
[ssf2_rl] Game is up; bridge listening on 127.0.0.1:4567.
stage: battlefield
P1: external agent (step-driven), character=marth
P2: Python ZeroBot (zero), character=zamus

frame: 1 | me: Marth | opp: Zero Suit Samus
ZeroBot held controls: 0


In [12]:
# Sequential reset on the same environment: human versus native level-9 CPU.
obs, info = env.reset(
    players={1: Human(Character.Marth), 2: CPU(Character.Samus, level=9)},
    stage=Stage.bf,
)
print(env.describe_matchup())
env.run(frames=20 * 30)
print("Completed 600 streamed frames without a bridge timeout.")

stage: battlefield
P1: human, character=marth
P2: in-game CPU level 9, character=samus
run(): 600 frames, 0 dropped
Completed 600 streamed frames without a bridge timeout.


In [ ]:
# --- Scripted bot (dashdance) vs ZeroBot, 450 frames -------------------------
# render_controls=1 shows P1's held inputs in the game window (bottom-left).
script = [
    (NOOP, 100),
    (DOWN | SPECIAL, 10),
    (LEFT, 20),
    (RIGHT, 20),
    (LEFT, 20),
    (RIGHT, 20),
]

obs, info = env.reset(
    players={
        1: ScriptedBot(Character.Marth, script, on_end="loop"),
        2: ZeroBot(Character.Samus),
    }
    # render_controls=1,  # watch P1's controls in the AIR window
)
print(env.describe_matchup())
traj = env.run(frames=450, record=True)

# The last Python mask remains held after run() returns; neither slot falls
# through to native AI.

stage: finaldestination
P1: Python ScriptedBot (scripted), character=marth
P2: Python ZeroBot (zero), character=samus
run(): 450 frames, 5696 dropped


In [14]:
# --- Fast exact lockstep -----------------------------------------------------
# Close the real-time client before opening the one supported active client.
env.close()
lockstep_env = SSF2Env(
    lockstep=True,
    lockstep_mode="synchronous",
    state_transport="json",  # compatibility default; benchmark separately
)
obs, info = lockstep_env.reset(
    players={
        1: Agent(Character.Marth),
        2: ZeroBot(Character.Samus),
    },
    stage=Stage.bf,
)
assert info["lockstep"] and info["paused"]
previous = info["frame"]

for _ in range(301):
    obs, reward, terminated, truncated, info = lockstep_env.step(0)
    assert info["paused"] and info["frame"] == previous + 1
    previous = info["frame"]

print(f"Paused at frame {info['frame']} after 301 exact policy steps.")
lockstep_env.close()

Paused at frame 302 after 301 exact policy steps.


In [5]:
# --- Human vs FollowBot (observation sanity check) ---------------------------
# Reconnect the reusable real-time environment after closing lockstep_env.
obs, info = env.reset(
    players={
        1: Human(Character.Marth),
        2: FollowBot(Character.Samus, deadzone=30.0)
    },
    render_controls=1

)
print(env.describe_matchup())
traj = env.run(frames=600, record=True)

# FollowBot's final mask remains held after run() returns instead of reverting
# to native CPU behavior.

stage: finaldestination
P1: human, character=marth
P2: Python FollowBot (follow), character=samus
run(): 600 frames, 5696 dropped
